Step 1: Data Preprocessing

In [1]:
import pandas as pd
import numpy as np
import torch
import random
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from torch.utils.data import DataLoader, TensorDataset

# Set a fixed random seed for reproducibility
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

def create_8x8_matrix(data_row):
    numeric_row = pd.to_numeric(data_row, errors='coerce').fillna(0)
    num_elements_required = 64
    if len(numeric_row) < num_elements_required:
        numeric_row = np.pad(numeric_row, (0, num_elements_required - len(numeric_row)), 'constant')
    matrix = np.array(numeric_row).reshape(8, 8)
    return matrix

def process_dataset(dataset):
    matrices = []
    for _, row in dataset.iterrows():
        matrix = create_8x8_matrix(row)
        matrices.append(matrix)
    return matrices

# Load dataset
df = pd.read_csv('kddcup.data_10_percent.csv', header=None)

# Identify and remove non-numeric columns
non_numeric_columns = [1, 2, 3]
df_features = df.drop(non_numeric_columns, axis=1)

# Process each row to create 8x8 matrices
matrices = process_dataset(df_features)

# Convert to a 3D numpy array
X = np.array(matrices)

# Encode the labels
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(df.iloc[:, -1])

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=SEED)

# Convert to PyTorch tensors
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.long)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.long)


Step 2: Building the Model

In [8]:
import torch
import torch.nn as nn

class DeepAutoencoder(nn.Module):
    def __init__(self, num_input_features, num_classes, dropout_rate=0.4):
        super(DeepAutoencoder, self).__init__()
        self.num_input_features = num_input_features

        # Adjust input size for Conv1d with dilation=3
        conv1d_output_size_dilated3 = (num_input_features - 3 * (3 - 1) - 1) + 1  # Adjusted for dilation=3

        # Dilated Encoder with dilation factor of 3
        self.encoder_dilated3 = nn.Sequential(
            nn.Linear(num_input_features, 64),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Conv1d(in_channels=1, out_channels=1, kernel_size=3, stride=1, dilation=3),  # Adjusted dilation
            nn.Flatten(),
            nn.Linear(conv1d_output_size_dilated3, 32)
        )

        # Dilated Decoder with adjusted input and output sizes for dilation=3
        self.decoder_dilated3 = nn.Sequential(
            nn.Linear(32, conv1d_output_size_dilated3),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Unflatten(dim=1, unflattened_size=(1, conv1d_output_size_dilated3)),  # Ensure correct input shape for ConvTranspose1d
            nn.ConvTranspose1d(1, 1, kernel_size=3, stride=1, dilation=3),  # Adjusted dilation
            nn.Flatten(),
            nn.Linear(64, num_input_features)
        )

        # Classifier
        self.classifier = nn.Linear(32, num_classes)

    def forward(self, x):
        x = x.view(-1, self.num_input_features)
        encoded_dilated3 = self.encoder_dilated3(x.unsqueeze(1))
        decoded_dilated3 = self.decoder_dilated3(encoded_dilated3)
        classification = self.classifier(encoded_dilated3)
        return decoded_dilated3, classification



# Example usage
num_classes = len(label_encoder.classes_)  # Update this based on your dataset
print(num_classes)
num_input_features = 64  # Update this based on your dataset
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = DeepAutoencoder(num_input_features, num_classes).to(device)


Step 3: Training the Model

In [9]:
import torch.optim as optim  # Import the optim module

LEARNING_RATE = 0.001  # Adjusted learning rate
BATCH_SIZE = 64        # Adjusted batch size

criterion_reconstruction = nn.L1Loss()
criterion_classification = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

train_data = TensorDataset(X_train_tensor, y_train_tensor)
train_loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True)

num_epochs = 100



for epoch in range(num_epochs):
    model.train()
    running_loss_reconstruction = 0.0
    running_loss_classification = 0.0

    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()
        decoded, classification = model(inputs.view(-1, num_input_features))
        loss_reconstruction = criterion_reconstruction(decoded, inputs.view(-1, num_input_features))
        loss_classification = criterion_classification(classification, labels)
        loss = loss_reconstruction + loss_classification
        loss.backward()
        optimizer.step()

        running_loss_reconstruction += loss_reconstruction.item()
        running_loss_classification += loss_classification.item()

    print(f'Epoch [{epoch+1}/{num_epochs}], Reconstruction Loss: {running_loss_reconstruction / len(train_loader)}, Classification Loss: {running_loss_classification / len(train_loader)}')

Epoch [1/100], Reconstruction Loss: 70.69995840546999, Classification Loss: 32.332977359665385
Epoch [2/100], Reconstruction Loss: 44.94294517554973, Classification Loss: 34.9073727769252
Epoch [3/100], Reconstruction Loss: 47.556468833913456, Classification Loss: 6.93417554052395
Epoch [4/100], Reconstruction Loss: 54.49947083362849, Classification Loss: 19.248534245163093
Epoch [5/100], Reconstruction Loss: 49.1660456616232, Classification Loss: 11.347459560651224
Epoch [6/100], Reconstruction Loss: 43.09464437125672, Classification Loss: 10.38052367407741
Epoch [7/100], Reconstruction Loss: 35.8662166790641, Classification Loss: 11.166995173458625
Epoch [8/100], Reconstruction Loss: 31.205009534939585, Classification Loss: 3.421536663291286
Epoch [9/100], Reconstruction Loss: 55.26905227843962, Classification Loss: 26.708868695809752
Epoch [10/100], Reconstruction Loss: 31.331052902696047, Classification Loss: 5.929865550561905
Epoch [11/100], Reconstruction Loss: 52.286529940850585

Step 4: Evaluating the Model and Calculating Metrics

In [10]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
import csv

model.eval()
true_labels = []
predicted_labels = []

test_data = TensorDataset(X_test_tensor, y_test_tensor)
test_loader = DataLoader(test_data, batch_size=BATCH_SIZE, shuffle=False)

with torch.no_grad():
    for inputs, labels in test_loader:
        inputs = inputs.view(-1, num_input_features)
        inputs, labels = inputs.to(device), labels.to(device)
        decoded, classification = model(inputs)
        _, predicted = torch.max(classification, 1)
        true_labels.extend(labels.cpu().tolist())
        predicted_labels.extend(predicted.cpu().tolist())

accuracy = accuracy_score(true_labels, predicted_labels)
precision = precision_score(true_labels, predicted_labels, average='weighted', zero_division=1)
recall = recall_score(true_labels, predicted_labels, average='weighted', zero_division=1)
f1 = f1_score(true_labels, predicted_labels, average='weighted', zero_division=1)
conf_matrix = confusion_matrix(true_labels, predicted_labels)

print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)
print("F1 Score:", f1)
print("Confusion Matrix:\n", conf_matrix)

csv_file = "kdd99d3o.csv"

# نوشتن داده به فایل CSV
with open(csv_file, mode='w', newline='', encoding='utf-8-sig') as file:
    writer = csv.writer(file)
    writer.writerows(conf_matrix)

print(f"فایل {csv_file} با موفقیت ایجاد شد.")

Accuracy: 0.982834842031966
Precision: 0.9819450360725827
Recall: 0.982834842031966
F1 Score: 0.9766308117981666
Confusion Matrix:
 [[  538     0     0     0     0     0     0     0     0     0     0     3
      0     0     0     0     0     0     0     0     0]
 [    0     0     0     0     0     0     0     0     0     0     0    11
      0     0     0     0     0     0     0     0     0]
 [    0     0     0     0     0     0     0     0     0     0     0     1
      0     0     0     0     0     0     0     0     0]
 [    0     0     0     0     0     0     0     0     0     0     0     9
      0     0     0     0     0     1     0     0     0]
 [    0     0     0     0     0     0     0     0     0     0     0     3
      0     0     0     0     0     0     0     0     0]
 [    0     0     0     0     0     0     0     0     0    52     0   268
      0     0     0     0     0     4     0     0     0]
 [    0     0     0     0     0     0     0     0     0     5     0     0
      0 

In [1]:
import csv

# آرایه‌ی داده جدید
new_data = [
    [538, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 11, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 9, 0, 0, 0, 0, 0, 1, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 0, 0, 0, 52, 0, 268, 0, 0, 0, 0, 0, 4, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 0, 0, 0, 5, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 0, 0, 0, 26706, 0, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 0, 0, 0, 22, 0, 30, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 0, 0, 0, 73, 0, 23816, 0, 0, 0, 0, 0, 307, 0, 44, 27],
    [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 46, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 0, 0, 0, 26, 0, 248, 0, 0, 0, 0, 0, 13, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 0, 0, 0, 68, 0, 12, 0, 0, 0, 0, 0, 308, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 30, 0, 0, 0, 0, 0, 70306, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 0, 0, 0, 231, 0, 0, 0, 0, 0, 0, 0, 9, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 260, 0, 0, 0, 0, 0, 0, 0, 14, 0],
    [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 6]
]

# نام فایل CSV
csv_file_new = "kdd99d3o.csv"

# نوشتن داده‌های جدید به فایل CSV
with open(csv_file_new, mode='w', newline='', encoding='utf-8-sig') as file:
    writer = csv.writer(file)
    writer.writerows(new_data)

print(f"فایل {csv_file_new} با موفقیت ایجاد شد.")


فایل kdd99d3o.csv با موفقیت ایجاد شد.
